# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #3 - Click Capture by Position Tier (marked CONFIRMED)**

Where does the label come from? Weighted CTR = total clicks divided by total impressions, summed across the whole position tier, not an average of individual page CTRs. The paper mentions they fixed an older per-row-average version that produced impossible values above 100%, which is a good sign they already caught one of their own bugs.

Does the validation design carry the claim? This is a cross-sectional snapshot (one period, no before/after), so it can support "we observed this pattern" but not "refining titles will improve CTR," which is a causal claim. The paper's own recommended next step ("measure over the next 30 days") actually 
matches this honestly, they're framing it as decision-support, not proven causation.

**ML Appendix - What Predicts Health? (Random Forest feature importance)**

Where does the label come from? Health score, which the paper itself admits is "partly constructed from some of these inputs", meaning some of the model's features are also ingredients of the label which it is predicting.

Does the validation design carry the claim? Not fully. If Average Position and Impressions are both inputs to health score and the top two most "important" predictors of it, that's the same circularity I found in my ML-08 baseline, the model is partly just reading back pieces of its own label. The paper does flag this itself ("importance is descriptive rather than causal"), which is the honest move, but it's worth naming explicitly rather than skimming past it.

In [13]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()

tier_median_ctr = visible.groupby('position_tier')['ctr'].transform('median')
visible['ctr_gap'] = tier_median_ctr - visible['ctr']

print(f"visible pages (n): {len(visible):,}")

visible pages (n): 12,023


In [14]:
weighted_ctr = visible.groupby('position_tier').apply(
    lambda g: g['clicks_90d'].sum() / g['impressions_90d'].sum()
)
tier_order = ['top_3', 'page_1', 'striking']
print(weighted_ctr.reindex(tier_order))

position_tier
top_3       0.004879
page_1      0.003497
striking    0.003491
dtype: float64


**Comparing my data to Finding #3:**

My weighted CTR: top_3 = 0.49%, page_1 = 0.35%, striking = 0.35% 

Paper's weighted CTR: top_3 = 0.423%, page_1 = 0.339%, striking = 0.325%

My numbers are **close** to the paper's, and using this weighted method (total clicks divided by total impressions), the order comes out clean: top_3 > page_1 > striking, the same ranking claimed in Finding #3.

This is different from my own Signal A in ML-07, where using median CTR per page, top_3 actually came out lower than page_1. 

**The difference is the method**: weighted CTR gives more influence to pages with more traffic, while my earlier median approach gave every page equal weight regardless of traffic, including my small top_3 group (only 458 pages). This tells me Finding #3 holds up under the paper's stated method, and my earlier MIXED result was likely a sample-size artifact from a less robust method, not a real contradiction.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**My split, before/after**: I already built a grouped-by-client split in ML-08 to prevent the model from memorizing a client's specific pages. Here I'm re-confirming it side by side with what a plain random split would have looked like, to show the honest split actually matters.

In [15]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

feature_cols = [
    'search_volume', 'competition', 'cpc', 'content_type', 'main_intent',
    'word_count', 'char_count', 'provider_used', 'model_used',
    'impressions_90d', 'days_with_impressions', 'content_age_days',
    'days_since_last_update', 'avg_position', 'position_tier',
    'impression_tier', 'freshness_tier', 'age_tier', 'word_count_tier', 'char_count_tier'
]
categorical_cols = ['content_type', 'main_intent', 'provider_used', 'model_used',
                     'position_tier', 'impression_tier', 'freshness_tier',
                     'age_tier', 'word_count_tier', 'char_count_tier']
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

gap_threshold = visible['ctr_gap'].quantile(0.75)
visible['needs_review'] = (visible['ctr_gap'] >= gap_threshold).astype(int)

X = visible[feature_cols].copy()
X[numeric_cols] = X[numeric_cols].fillna(0)
X[categorical_cols] = X[categorical_cols].fillna('unknown').astype(str)
y = visible['needs_review']

def build_and_score(train_idx, test_idx, label):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    prep = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ])
    pipe = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
    pipe.fit(Xtr, ytr)
    probs = pipe.predict_proba(Xte)[:, 1]
    res = visible.iloc[test_idx].copy()
    res['model_score'] = probs
    K = 50
    p_at_k = res.sort_values('model_score', ascending=False).head(K)['needs_review'].mean()
    print(f"{label}: precision@{K} = {p_at_k:.3f}  (test n={len(Xte)})")
    return p_at_k

# RANDOM split (not honest)
rand_train_idx, rand_test_idx = train_test_split(range(len(X)), test_size=0.2, random_state=42)
random_score = build_and_score(rand_train_idx, rand_test_idx, "RANDOM split")

# GROUPED split (honest - same as ML-08)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
grp_train_idx, grp_test_idx = next(splitter.split(X, y, groups=visible['client_id']))
grouped_score = build_and_score(grp_train_idx, grp_test_idx, "GROUPED split (honest)")

print(f"\ngap between random and grouped: {random_score - grouped_score:.3f}")

RANDOM split: precision@50 = 0.720  (test n=2405)
GROUPED split (honest): precision@50 = 0.400  (test n=821)

gap between random and grouped: 0.320


**Before/after: random split vs. grouped split**

Random split: precision@50 = 0.720

Grouped split (honest): precision@50 = 0.400

Gap: 0.320

This is a big gap, and it is not a good sign for the random split. It's a warning sign. When the same client's pages can appear in both train and test, the model partly memorizes that specific client's patterns instead of learning something new or general. Then, when it's "tested" on more pages from a client it already saw in training, it looks like it's doing great, but that's not real skill, it's 
leakage through repetition.

If I had only run the random split, I would have reported 0.720 as my model's performance, which would have been a real overstatement, not because I lied, but because the test itself wasn't honest. The grouped number (0.400) is the one I actually trust, because it tests the model on clients it has genuinely never seen before, which is the real situation a deployed model would face.

This confirms my ML-08 split design was the right call. Without it, I'd have reported a number nearly double what the model actually deserves.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage audit on my final feature set.** I'm running the same hunt from ML-04's leak trap, but this time specifically on the 20 features I actually used in my ML-08 model, checking each against the leakage taxonomy: label-derived features, future/overlapping windows, and decision-derived (product) features.

In [16]:
label_derived = {'ctr', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'ctr_gap', 'needs_review'}
future_overlapping = {'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d',
                       'ai_sessions_90d', 'scroll_events_90d', 'engagement_rate', 'scroll_rate',
                       'ai_traffic_pct', 'days_with_sessions', 'trend_direction', 'trend_pct'}
product_flags = {'health_score', 'priority_score', 'action_type', 'needs_ctr_fix', 'is_quick_win'}

used_features = set(feature_cols)

print("Checking my 20 ML-08 features against each leakage category:\n")
print("1. Label-derived overlap:", used_features & label_derived)
print("2. Future/overlapping-window overlap:", used_features & future_overlapping)
print("3. Product-flag overlap:", used_features & product_flags)
print()
print("Base rate (needs_review positive rate):", f"{y.mean():.2%}")
print("Model precision@50 (honest, grouped split):", f"{grouped_score:.3f}")

Checking my 20 ML-08 features against each leakage category:

1. Label-derived overlap: set()
2. Future/overlapping-window overlap: set()
3. Product-flag overlap: set()

Base rate (needs_review positive rate): 26.14%
Model precision@50 (honest, grouped split): 0.400


**Leakage audit results**:

All three checks came back empty (set()):
1. Label-derived overlap: none of my features overlap with ctr, clicks, or the label itself.
2. Future/overlapping-window overlap: none of my features overlap with post-click behavior columns or    trend_direction/trend_pct.
3. Product-flag overlap: none of FlyRank's real decision flags are in my feature list (they don't even exist in this CSV, confirmed in ML-08).

This confirms my ML-08 features are clean and there is nothing that would let the model quietly read the answer instead of learning a real pattern.

**Base rate check:** If I guessed randomly, I'd expect about 26.14% precision, since that's the actual share of pages that need review in this data. My honest, grouped-split model gets 40.0% precision@50, a real 14-point lift over random guessing, not an inflated number from leakage or from a lucky random split (like the 72.0% I saw in section 2, which I already know not to trust).

Put together, this tells me my model's 40% isn't an accident and isn't leakage, it's genuine.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


**Original claim:**

"My strongest pick" (content_8ba781dafa55 had real search 
demand, real visibility, and zero clicks).

**Where this claim sits on the ladder:**

I only have a cross-sectional snapshot, one page, one 90-day window, no before/after test. "Strongest pick" implied more confidence than that evidence can carry. It read close to a recommendation that a fix would definitely work, which nothing in my data actually shows.

**Honest rewrite:** 

"This page shows a large observed CTR gap relative to its position tier, combined with real search demand and high impressions. It looks worth reviewing first, because the combination of visibility and zero clicks is unusual enough to investigate. This is decision-support, not a guarantee: I 
haven't tested whether a title change would actually improve its CTR."

**What changed:** 

"strongest pick" became "looks worth reviewing first, because " and I added an explicit line admitting I haven't tested the fix. That's the honest gap between what my data shows (an  unusual pattern) and what a causal claim would require (a before/after test I haven't run).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.